# 4.3 — Calculate the NumPy/SciPy and JAX comparison

Open this notebook in **Google Colab with a standard CPU runtime**. Calculations run on
**your personal Modal account**; no local GC2D checkout or B2 account is needed.
Public B2 provides verified, frozen scientific inputs and numerical sources.
Results are offered as a **ZIP download to your computer** with paired trajectories,
individual timings, parameters, library versions, checksums and source provenance.

Create a [Modal account](https://modal.com/) and a personal API token in its settings.
Modal compute uses your account's allowance and billing; check the
[current pricing](https://modal.com/pricing) before starting a full run.
No author's credentials are included. The notebook guides installation and authentication.

The reference compares **BM4Implicit**, **GaussLegendre4** and **RK2** for **10, 30 and
90 cycles**, using the 16-particle union of sets 3 and 4, normalized rho=0.015,
reflected potential, 40 fixed steps and ten saved samples per cycle, and float64.
One first call and three warmed calls per backend in nine cases give **72 integrations**.
Both backends run sequentially in the same two-core Modal CPU container for each case
(4 GiB requested, 8 GiB limit), alternating their order. Cases run sequentially;
JAX caches are cleared before every case, and repeated trajectories must match bitwise.

Timings include execution preparation, integration, synchronization and host result
construction. They exclude input restoration, startup, checks, files and transfers.
First JAX calls include compilation. Backend distances measure agreement in unwrapped
coordinates and **do not establish integration accuracy**. Frozen sources and dependency
versions reproduce the protocol; timings vary with hardware and runtime conditions.

To inspect the saved reference without calculating, open the
[visualization notebook](https://colab.research.google.com/drive/1xVRKtB2AWp_Pqv6COtP1F3z0Ystfa2Bo). Run the following cells in order.


## 1. Install the Modal client
Integrations run in the remote container. Colab also uses NumPy to audit the returned arrays.


In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    installed = importlib.metadata.version("modal")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "1.6.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "modal==1.6.0"], check=True)
try:
    import numpy
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "numpy"], check=True)
import modal
print("Modal client ready.")


## 2. Authenticate with your personal Modal account

Add `MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET` in Colab's **Secrets** panel (key icon),
and grant this notebook access. Hidden prompts are available as a fallback.
Do not paste credentials into code cells. They remain in this runtime's environment;
the notebook does not write them to saved outputs or result files.
Existing environment credentials are reused when running locally.


In [ ]:
import getpass
import os

for key in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    if not os.environ.get(key):
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(key)
        except Exception:
            pass
        if not value:
            value = getpass.getpass(f"{key}: ")
        if not value:
            raise ValueError("Both personal Modal credentials are required.")
        os.environ[key] = value.strip()
        del value
print("Credentials loaded for this runtime; no calculation has started.")


## 3. Choose the calculation

Keep `CYCLES = [10, 30, 90]` and `REPETITIONS = 3` for the reference protocol.
Set `CYCLES = [1]` for a short workflow check across all three methods; its timings
are not thesis results. A custom selection can contain up to three distinct durations
from 1 to 90 cycles. Set `LAUNCH_CALCULATION = True` when ready.

Keep the generated `RUN_ID` and download the receipt. Choose a new identifier before
starting a different experiment. After a disconnected session, enter your old
`RUN_ID`, the same settings and `RESUME_CALL_ID` to retrieve the existing call.
Modal call results have limited retention: download promptly. Colab runtime storage
is temporary, and saving the notebook does not save its result ZIP.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

CYCLES = [10, 30, 90]
REPETITIONS = 3  # @param {type:"integer"}
LAUNCH_CALCULATION = False  # @param {type:"boolean"}
RUN_ID = globals().get("RUN_ID", "reader-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + uuid4().hex[:6])
RESUME_CALL_ID = ""  # @param {type:"string"}
WORKSPACE = Path.cwd() / "gc2d-colab-43"
WORKSPACE.mkdir(exist_ok=True)
print(f"Run: {RUN_ID}; durations: {CYCLES}; {6 * len(CYCLES) * (REPETITIONS + 1)} integrations.")


## 4. Load and verify the reproduction kit

Download approximately 3.8 MB from public B2. Verify the pinned SHA-256
before loading archived Python inputs. Reviewed helper code is embedded below so
this notebook needs no checkout. Only the historical source-path provenance is
normalized for publication; numerical arrays, methods and execution settings are preserved.
The kit records the original and public input fingerprints and strips personal tar headers.


In [ ]:
import importlib
import json
import sys

KIT_URL = 'https://f003.backblazeb2.com/file/gc2d-colab-public/04_03_numpy_jax/calculation/v1/calculation-kit.tar.xz'
KIT_SHA256 = '33c91014d7ed2213bed2a8565f72822c51c02cc714c0d05304ae8ac7997ff446'
(WORKSPACE / "calculation_runtime.py").write_text('"""Portable input verification, submission recovery and paired-result checks."""\nfrom hashlib import sha256\nimport io\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport re\nimport tarfile\nimport tempfile\nfrom urllib.request import urlopen\nimport zipfile\n\nMETHODS = (\'BM4Implicit\', \'GaussLegendre4\', \'RK2\')\nBACKENDS = (\'scipy\', \'jax\')\nREFERENCE_CYCLES = (10, 30, 90)\nREFERENCE_RUN_ID = \'methods-20261119-union16-rho0015-c10-30-90\'\n\n\ndef check_settings(cycles, repetitions, run_id):\n    if (not isinstance(cycles, (tuple, list)) or not 1 <= len(cycles) <= 3\n            or any(type(c) is not int or not 1 <= c <= 90 for c in cycles)\n            or len(set(cycles)) != len(cycles)):\n        raise ValueError(\'Choose one to three distinct integer durations from 1 to 90 cycles.\')\n    if type(repetitions) is not int or not 3 <= repetitions <= 10:\n        raise ValueError(\'Choose 3 to 10 warmed repetitions.\')\n    if not isinstance(run_id, str) or not re.fullmatch(r\'[A-Za-z0-9][A-Za-z0-9_-]{0,79}\', run_id):\n        raise ValueError(\'Use a short run identifier containing letters, numbers, - or _.\')\n    return dict(cycles=list(cycles), repetitions=repetitions, run_id=run_id)\n\n\ndef safe_name(name):\n    path = PurePosixPath(name)\n    return bool(name) and not path.is_absolute() and \'..\' not in path.parts and \'\\\\\' not in name and str(path) == name\n\n\ndef unpack_tar(data, destination):\n    """Reject links, traversal and duplicate entries before writing any file."""\n    with tarfile.open(fileobj=io.BytesIO(data), mode=\'r:*\') as archive:\n        members, seen = archive.getmembers(), set()\n        for member in members:\n            target = destination / member.name\n            if (not member.isfile() or not safe_name(member.name) or member.name in seen\n                    or not target.resolve().is_relative_to(destination.resolve()) or target.is_symlink()):\n                raise ValueError(\'Unsafe or duplicated archive entry.\')\n            seen.add(member.name)\n        for member in members:\n            target = destination / member.name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(archive.extractfile(member).read())\n\n\ndef verify_inputs(kit):\n    manifest = json.loads((kit / \'inputs.json\').read_text())\n    cases = [f\'{m}-{c}\' for c in REFERENCE_CYCLES for m in METHODS]\n    expected = {\'source.tar.gz\', \'study.json\'} | {case + suffix for case in cases for suffix in (\'.json\', \'.payload\')}\n    if set(manifest[\'files\']) != expected or manifest[\'reference_run_id\'] != REFERENCE_RUN_ID:\n        raise ValueError(\'Unexpected reproduction-kit contents.\')\n    for name, entry in manifest[\'files\'].items():\n        data = (kit / name).read_bytes()\n        if len(data) != entry[\'bytes\'] or sha256(data).hexdigest() != entry[\'sha256\']:\n            raise ValueError(f\'Damaged input: {name}\')\n    for case in cases:\n        if manifest[\'provenance\'][case][\'public_payload_sha256\'] != manifest[\'files\'][case + \'.payload\'][\'sha256\']:\n            raise ValueError(\'Input provenance mismatch.\')\n    return manifest\n\n\ndef prepare_kit(data, expected_sha256, destination):\n    if sha256(data).hexdigest() != expected_sha256:\n        raise ValueError(\'Reproduction-kit checksum mismatch; nothing was executed.\')\n    unpack_tar(data, destination)\n    manifest = verify_inputs(destination)\n    unpack_tar((destination / \'source.tar.gz\').read_bytes(), destination / \'source\')\n    return manifest\n\n\ndef download_kit(url, expected_sha256, destination):\n    with urlopen(url, timeout=120) as response:\n        return prepare_kit(response.read(), expected_sha256, destination)\n\n\ndef write_json(path, record):\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(record, indent=2, sort_keys=True, allow_nan=False) + \'\\n\')\n    temporary.replace(path)\n\n\ndef launch_or_recover(worker, receipt_path, settings, kit_sha256, resume_call_id=\'\'):\n    """Record an ambiguous submission before spawn so it cannot silently repeat."""\n    requested = {**settings, \'kit_sha256\': kit_sha256}\n    if receipt_path.exists():\n        receipt = json.loads(receipt_path.read_text())\n        if receipt[\'settings\'] != requested:\n            raise ValueError(\'This run identifier has different settings; choose a new one.\')\n        if receipt[\'submission\'] != \'confirmed\':\n            raise RuntimeError(\'Submission was interrupted. Inspect your Modal dashboard before starting another run.\')\n        if resume_call_id and resume_call_id != receipt[\'call_id\']:\n            raise ValueError(\'The saved call differs from RESUME_CALL_ID.\')\n        return receipt\n    receipt_path.parent.mkdir(parents=True, exist_ok=True)\n    if resume_call_id:\n        if not re.fullmatch(r\'fc-[A-Za-z0-9]+\', resume_call_id):\n            raise ValueError(\'Expected a Modal call identifier beginning with fc-.\')\n        receipt = dict(submission=\'confirmed\', call_id=resume_call_id, settings=requested)\n        write_json(receipt_path, receipt)\n        return receipt\n    import modal\n    with modal.enable_output(), worker.app.run(detach=True):\n        write_json(receipt_path, dict(submission=\'unconfirmed\', settings=requested))\n        call = worker.calculate.spawn(**settings)\n        receipt = dict(submission=\'confirmed\', call_id=call.object_id, settings=requested)\n        write_json(receipt_path, receipt)\n    return receipt\n\n\ndef verify_results(data, settings, original_inputs=None):\n    """Check every ZIP entry and paired timing audit before offering a download."""\n    import numpy as np\n    with zipfile.ZipFile(io.BytesIO(data)) as archive:\n        names = archive.namelist()\n        if len(names) != len(set(names)) or any(not safe_name(n) for n in names):\n            raise ValueError(\'Unsafe result archive.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if set(names) != set(manifest[\'files\']) | {\'manifest.json\'}:\n            raise ValueError(\'Result manifest does not cover the complete ZIP.\')\n        for name, entry in manifest[\'files\'].items():\n            value = archive.read(name)\n            if len(value) != entry[\'bytes\'] or sha256(value).hexdigest() != entry[\'sha256\']:\n                raise ValueError(f\'Damaged result: {name}\')\n        study = json.loads(archive.read(\'study.json\'))\n        if any(study[k] != v for k, v in settings.items()):\n            raise ValueError(\'The recovered call used different settings.\')\n        if (study[\'methods\'] != list(METHODS) or study[\'particle_count\'] != 16\n                or study[\'rho\'] != .015 or study[\'reference_run_id\'] != REFERENCE_RUN_ID):\n            raise ValueError(\'Wrong methods, particles or reference study.\')\n        inputs = json.loads(archive.read(\'inputs.json\'))\n        if original_inputs is not None and inputs != original_inputs:\n            raise ValueError(\'The recovered call used different original inputs.\')\n        if sha256(archive.read(\'source.tar.gz\')).hexdigest() != inputs[\'files\'][\'source.tar.gz\'][\'sha256\']:\n            raise ValueError(\'Different frozen sources in returned archive.\')\n        timings = json.loads(archive.read(\'timings.json\'))\n        expected_cases = {f\'{m}-{c}\' for c in settings[\'cycles\'] for m in METHODS}\n        if set(timings[\'cases\']) != expected_cases:\n            raise ValueError(\'Missing or unexpected paired cases.\')\n        expected_files = {\'study.json\', \'inputs.json\', \'source.tar.gz\', \'timings.json\', \'manifest.json\'}\n        for case, pair in timings[\'cases\'].items():\n            method, count_string = case.rsplit(\'-\', 1)\n            count = int(count_string)\n            template = f\'{method}-{count if count in REFERENCE_CYCLES else 10}\'\n            original_sha = inputs[\'files\'][template + \'.payload\'][\'sha256\']\n            if pair[\'original_payload_sha256\'] != original_sha or (count in REFERENCE_CYCLES and pair[\'payload_sha256\'] != original_sha):\n                raise ValueError(\'Different original numerical payload.\')\n            if pair[\'reference_payload_sha256\'] != inputs[\'provenance\'][template][\'payload_sha256\']:\n                raise ValueError(\'Different historical input provenance.\')\n            if not pair[\'environment\'][\'x64\'] or not pair[\'same_container\'] or not pair[\'repeated_states_identical\']:\n                raise ValueError(\'Worker audits failed.\')\n            rows = pair[\'measurements\']\n            expected = [(b, r) for r in range(settings[\'repetitions\'] + 1)\n                for b in (BACKENDS if r % 2 == 0 else BACKENDS[::-1])]\n            if [(r[\'backend\'], r[\'repetition\']) for r in rows] != expected:\n                raise ValueError(\'Incomplete, duplicated or incorrectly ordered measurements.\')\n            for row in rows:\n                first = row[\'repetition\'] == 0\n                cache = (1, 0) if first else (0, 1)\n                if row[\'backend\'] == \'scipy\':\n                    cache = (0, 0)\n                if (row[\'integration_cache_misses\'], row[\'integration_cache_hits\']) != cache:\n                    raise ValueError(\'Invalid compilation-cache audit.\')\n                if row[\'phase\'] != (\'first\' if first else \'warm\') or not math.isfinite(row[\'seconds\']) or row[\'seconds\'] <= 0:\n                    raise ValueError(\'Invalid timing measurement.\')\n                if not math.isfinite(row[\'process_seconds\']) or row[\'process_seconds\'] < 0:\n                    raise ValueError(\'Invalid process time.\')\n            for metric in (\'maximum_distance\', \'rms_distance\', \'final_maximum_distance\', \'maximum_distance_first_10_cycles\'):\n                if not math.isfinite(pair[\'agreement\'][metric]) or pair[\'agreement\'][metric] < 0:\n                    raise ValueError(\'Invalid backend discrepancy.\')\n            pair_name = f\'runs/{case}/pair.json\'\n            if json.loads(archive.read(pair_name)) != pair:\n                raise ValueError(\'Inconsistent paired timing records.\')\n            expected_files.add(pair_name)\n            states, times = {}, {}\n            for backend in BACKENDS:\n                base = f\'runs/{case}/{backend}/\'\n                expected_files.update(base + f for f in (\'solution.npz\', \'metadata.json\', \'manifest.json\'))\n                metadata = json.loads(archive.read(base + \'metadata.json\'))\n                experiment, diagnostics = metadata[\'experiment\'], metadata[\'diagnostics\']\n                if (experiment[\'method\'] != method or experiment[\'backend\'] != backend\n                        or experiment[\'run_id\'] != settings[\'run_id\'] or experiment[\'config\'][\'cycles\'] != count\n                        or diagnostics[\'step_count\'][\'scalar\'] != count * 40\n                        or diagnostics[\'output_interpolation_count\'][\'scalar\'] != 0):\n                    raise ValueError(\'Incorrect trajectory metadata or step count.\')\n                with np.load(io.BytesIO(archive.read(base + \'solution.npz\')), allow_pickle=False) as arrays:\n                    states[backend], times[backend] = arrays[\'states\'].copy(), arrays[\'t\'].copy()\n                if (states[backend].shape != (count * 10 + 1, 32) or states[backend].dtype != np.float64\n                        or not np.isfinite(states[backend]).all()\n                        or times[backend].shape != (count * 10 + 1,)\n                        or not np.allclose(times[backend], np.linspace(0, count, count * 10 + 1), rtol=0, atol=2e-12)):\n                    raise ValueError(\'Incomplete or invalid numerical trajectory.\')\n            np.testing.assert_array_equal(times[\'scipy\'], times[\'jax\'])\n            np.testing.assert_array_equal(states[\'scipy\'][0], states[\'jax\'][0])\n            delta = states[\'scipy\'] - states[\'jax\']\n            distance = np.hypot(delta[:, :16], delta[:, 16:])\n            actual = dict(maximum_distance=float(distance.max()), rms_distance=float(np.sqrt(np.mean(distance**2))),\n                final_maximum_distance=float(distance[-1].max()), maximum_distance_first_10_cycles=float(distance[:101].max()))\n            if any(not np.isclose(pair[\'agreement\'][k], v, rtol=1e-13, atol=0) for k, v in actual.items()):\n                raise ValueError(\'Backend agreement does not match the saved trajectories.\')\n        if set(names) != expected_files:\n            raise ValueError(\'Incomplete trajectories or unexpected result files.\')\n        return study, timings\n\n\ndef calculate_local(kit, cycles=(10, 30, 90), repetitions=3, run_id=\'reader-run\'):\n    """Run all selected pairs; also allow a disposable local smoke validation."""\n    from diagnostics.io.persistence import save_solution, load_solution\n    from execution._modal_worker import decode_job, encode_job\n    from request import SimulationRequest\n    from studies.bm4_numpy_jax import benchmark_pair\n    import numpy as np\n\n    settings = check_settings(cycles, repetitions, run_id)\n    inputs = verify_inputs(kit)\n    study = json.loads((kit / \'study.json\').read_text())\n    study.pop(\'bucket\', None)\n    study.update(settings, reference_run_id=inputs[\'reference_run_id\'],\n        storage=\'ZIP download to the reader computer\',\n        reference_configuration=list(cycles) == list(REFERENCE_CYCLES) and repetitions == 3,\n        execution_layout=\'All cases sequential; both backends share one container for each case; caches reset before every case\',\n        original_input_files=inputs[\'files\'])\n    timings = dict(schema_version=1, cases={})\n    with tempfile.TemporaryDirectory(prefix=\'gc2d-paired-results-\') as directory:\n        output = Path(directory)\n        for count in cycles:\n            for method in METHODS:\n                case = f\'{method}-{count}\'\n                template = f\'{method}-{count if count in REFERENCE_CYCLES else 10}\'\n                original_payload = (kit / (template + \'.payload\')).read_bytes()\n                payload = original_payload\n                if count not in REFERENCE_CYCLES:\n                    problem, numerical_method, options, observers = decode_job(payload)\n                    request = SimulationRequest.uniform(t_span=(0., float(count)), sample_count=count * 10 + 1)\n                    payload = encode_job(problem.with_request(request), numerical_method, options, observers)\n                # The original function resets both JAX and integration caches,\n                # alternates library order and asserts bitwise repeatability.\n                pair = benchmark_pair(payload, repetitions)\n                solutions = pair.pop(\'solutions\')\n                pair.update(original_payload_sha256=sha256(original_payload).hexdigest(),\n                    reference_payload_sha256=inputs[\'provenance\'][template][\'payload_sha256\'],\n                    same_container=True, repeated_states_identical=True)\n                timings[\'cases\'][case] = pair\n                metadata = json.loads((kit / (template + \'.json\')).read_text())\n                metadata[\'config\'][\'cycles\'] = count\n                metadata.update(repetitions=repetitions, run_id=run_id, pair=pair,\n                    payload_sha256=sha256(payload).hexdigest(), environment=pair[\'environment\'],\n                    reference_payload_sha256=pair[\'reference_payload_sha256\'],\n                    original_payload_sha256=sha256(original_payload).hexdigest())\n                for backend, solution in solutions.items():\n                    if (solution.states.shape != (count * 10 + 1, 32)\n                            or solution.diagnostics[\'step_count\'] != count * 40\n                            or solution.states.dtype != np.float64 or not np.isfinite(solution.states).all()\n                            or solution.t[0] != 0 or solution.t[-1] != count):\n                        raise ValueError(f\'Incomplete trajectory: {case}/{backend}\')\n                    target = output / \'runs\' / case / backend\n                    save_solution(solution, target, metadata={**metadata, \'backend\': backend})\n                    restored = load_solution(target)\n                    np.testing.assert_array_equal(restored.solution.t, solution.t)\n                    np.testing.assert_array_equal(restored.solution.states, solution.states)\n                (output / \'runs\' / case / \'pair.json\').write_text(json.dumps(pair, indent=2, allow_nan=False) + \'\\n\')\n        for name, value in [(\'study.json\', study), (\'timings.json\', timings), (\'inputs.json\', inputs)]:\n            (output / name).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n        (output / \'source.tar.gz\').write_bytes((kit / \'source.tar.gz\').read_bytes())\n        files = {p.relative_to(output).as_posix(): p.read_bytes() for p in sorted(output.rglob(\'*\')) if p.is_file()}\n        manifest = dict(schema_version=1,\n            files={name: dict(bytes=len(data), sha256=sha256(data).hexdigest()) for name, data in files.items()})\n        files[\'manifest.json\'] = (json.dumps(manifest, indent=2, sort_keys=True) + \'\\n\').encode()\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\', zipfile.ZIP_DEFLATED) as archive:\n            for name, data in files.items():\n                archive.writestr(name, data)\n        return buffer.getvalue()\n')
(WORKSPACE / "calculation_worker.py").write_text('"""Personal Modal worker for the frozen paired NumPy/SciPy and JAX study."""\nfrom pathlib import Path\n\nimport modal\n\nROOT = Path(__file__).resolve().parent\nimage = (modal.Image.debian_slim(python_version=\'3.12\')\n    .pip_install(\'numpy==1.26.4\', \'scipy==1.17.1\', \'h5py==3.16.0\',\n        \'threadpoolctl==3.6.0\', \'jax==0.7.1\', \'jaxlib==0.7.1\')\n    .env({\'PYTHONPATH\': \'/opt/gc2d/source/src\', \'JAX_ENABLE_X64\': \'true\',\n        \'JAX_PLATFORMS\': \'cpu\', \'JAX_ENABLE_COMPILATION_CACHE\': \'false\',\n        \'OMP_NUM_THREADS\': \'2\', \'OPENBLAS_NUM_THREADS\': \'2\', \'MKL_NUM_THREADS\': \'2\'})\n    .add_local_dir(ROOT / \'kit\', \'/opt/gc2d\')\n    .add_local_python_source(\'calculation_runtime\'))\napp = modal.App(\'gc2d-colab-numpy-jax\')\n\n\n\n\n@app.function(image=image, cpu=(2., 2.), memory=(4096, 8192), timeout=14400,\n    retries=0, max_containers=1, scaledown_window=60)\ndef calculate(cycles=(10, 30, 90), repetitions=3, run_id=\'reader-run\'):\n    from calculation_runtime import calculate_local\n    return calculate_local(Path(\'/opt/gc2d\'), cycles, repetitions, run_id)\n')
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
importlib.invalidate_caches()
import calculation_runtime as runtime
runtime = importlib.reload(runtime)
settings = runtime.check_settings(CYCLES, REPETITIONS, RUN_ID)
inputs = runtime.download_kit(KIT_URL, KIT_SHA256, WORKSPACE / "kit")
import calculation_worker as worker
worker = importlib.reload(worker)
print("Inputs and frozen sources verified. No calculation has started.")


## 5. Launch once, or recover the recorded call

Keep the receipt. A confirmed call is reused if this cell runs again.
If submission stops before recording a call ID, inspect your
[Modal dashboard](https://modal.com/apps) before starting another experiment.
This guard prevents automatic duplicate integrations after an ambiguous submission.


In [ ]:
from IPython.display import display, FileLink

if not LAUNCH_CALCULATION and not RESUME_CALL_ID:
    raise RuntimeError("Set LAUNCH_CALCULATION=True in cell 3, then run cells 3 to 7.")
RUN_DIRECTORY = WORKSPACE / "results" / RUN_ID
RECEIPT = RUN_DIRECTORY / "receipt.json"
receipt = runtime.launch_or_recover(worker, RECEIPT, settings, KIT_SHA256, RESUME_CALL_ID)
print(json.dumps(receipt, indent=2))
display(FileLink(str(RECEIPT)))
try:
    from google.colab import files
    files.download(str(RECEIPT))
except ImportError:
    pass


## 6. Retrieve and verify the completed result

This cell waits for the saved call and never launches a new one. A timeout can be
retried with the same receipt. If Modal reports an expired result, check any ZIP
already downloaded. The verified archive includes both trajectories per case,
paired timings, metadata, the frozen source and a complete checksum manifest.


In [ ]:
from statistics import median

RESULT_ZIP = RUN_DIRECTORY / (RUN_ID + ".zip")
if RESULT_ZIP.exists():
    result_bytes = RESULT_ZIP.read_bytes()
else:
    result_bytes = modal.FunctionCall.from_id(receipt["call_id"]).get(timeout=3600)
study, timings = runtime.verify_results(result_bytes, settings, inputs)
temporary_zip = RESULT_ZIP.with_suffix(".tmp")
temporary_zip.write_bytes(result_bytes)
temporary_zip.replace(RESULT_ZIP)
print(f"Verified {len(timings['cases'])} cases; result archive: {len(result_bytes):,} bytes.")
print(f"{'Case':24} {'Library':12} {'First (s)':>12} {'Median (s)':>12} {'Max distance':>14}")
for case, pair in timings["cases"].items():
    for backend in runtime.BACKENDS:
        rows = [r for r in pair["measurements"] if r["backend"] == backend]
        first = next(r["seconds"] for r in rows if r["repetition"] == 0)
        warm = [r["seconds"] for r in rows if r["repetition"] > 0]
        distance = pair["agreement"]["maximum_distance"]
        print(f"{case:24} {backend:12} {first:12.6f} {median(warm):12.6f} {distance:14.6e}")


## 7. Download to your computer

Allow the browser download and save the ZIP and receipt before closing the runtime.
No personal bucket or Modal Volume is needed. This reader-download destination is
deliberate: a reader has no write access to the author's scientific B2 archive.
The ZIP retains numerical arrays and metadata for your own analysis. The separate
visualization notebook displays the saved thesis reference without rerunning it.


In [ ]:
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    display(FileLink(str(RESULT_ZIP)))
print("Download offered. Check that the ZIP is saved on your computer.")
